# Aula 09 — Laboratório: busca aleatória e AutoML

**Consultoria Especializada em Ciência de Dados 1**

Na Aula 08 a grade testou todas as combinações que você escreveu. Hoje o orçamento passa a ser
**seu**: você diz quantas medições quer (`n_iter`) e sorteia onde medir. O laboratório treina as
três decisões que isso cria — **qual distribuição**, **qual faixa** e **quantos sorteios** — e
termina no AutoML, que faz a mesma coisa para o modelo inteiro.

A base é a mesma da Aula 08 (`5_musicas.csv`), com o mesmo alvo (`hit`), o mesmo Pipeline e o
mesmo teste guardado. O que muda é a busca.

Escreva as respostas nas células indicadas. Toda afirmação que menciona um número deve trazer o
número que saiu no seu notebook.

---
## Parte 0 — Ambiente, dados e o que já está montado

Nada aqui é novo: é o fecho da Aula 08. A coluna `popularidade` fica de fora (ela vaza o alvo —
voltamos a isso no Exercício 10) e o teste de 30 % fica guardado até o Exercício 8.

In [2]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

mus = pd.read_csv("5_musicas.csv")
y = mus["hit"]
X = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"]),
                   columns=["genero"])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def pipe():
    return Pipeline([("esc", StandardScaler()),
                     ("clf", RandomForestClassifier(n_estimators=100, random_state=0))])

print("X:", X.shape, "| hits:", int(y.sum()), f"({y.mean():.1%})")
print("treino:", len(X_tr), "| teste:", len(X_te), "| hits no teste:", int(y_te.sum()))
print("métrica da aula: average_precision (AP)")

X: (1050, 18) | hits: 128 (12.2%)
treino: 735 | teste: 315 | hits no teste: 38
métrica da aula: average_precision (AP)


---
## Parte 1 — Por que sortear, antes de sortear

Duas contas decidem a aula inteira: quanto custa cada caminho, e qual a chance de o sorteio
achar a região boa.

#### Exemplo

**Exemplo.** O custo da grade é o produto do número de valores de cada hiperparâmetro, vezes
as dobras. Com 2 valores de um e 3 de outro, e 5 dobras, são 2 × 3 × 5 = 30 ajustes.

In [3]:
valores = [2, 3]
combos = np.prod(valores)
print("combinações:", combos, "| com cv = 5:", combos * 5, "ajustes")

combinações: 6 | com cv = 5: 30 ajustes


### Exercício 1 — o custo de cada caminho
A floresta tem 5 hiperparâmetros. Uma grade modesta usa 4, 5, 4, 4 e 4 valores. Calcule:

1. quantas combinações a grade gera e quantos ajustes isso custa com `cv = 5`;
2. quantos ajustes custa uma `RandomizedSearchCV` com `n_iter = 40` e `cv = 5`;
3. quantos ajustes a grade custaria se você acrescentasse **um sexto** hiperparâmetro com 4
   valores — e quantos custaria a busca aleatória no mesmo caso.

In [5]:
# TODO — combinações da grade e ajustes com cv = 5
# TODO — ajustes da busca aleatória com n_iter = 40
# TODO — repita as duas contas com um sexto hiperparâmetro de 4 valores

**Resposta:**

1. O número de combinações da grade clássica é dado pelo produto dos valores de seus hiperparâmetros:
\(
4 \times 5 \times 4 \times 4 \times 4 = 1280 \text{ combinações}
\)
Com um esquema de validação cruzada com \( cv = 5 \), o custo total é de:
\(
1280 \times 5 = 6400 \text{ ajustes}
\)

2. Para a busca aleatória (`RandomizedSearchCV`), o custo total é fixo e depende exclusivamente do número de iterações definidos e das dobras de validação:
\(
40 \times 5 = 200 \text{ ajustes}
\)

3. Ao acrescentar o sexto hiperparâmetro com 4 valores:
* A grade clássica passaria a ter \( 1280 \times 4 = 5120 \text{ combinações} \), custando um total de \( 5120 \times 5 = 25600 \text{ ajustes} \).
* A busca aleatória manteria seu custo fixado no orçamento definido pelo usuário de \( 40 \times 5 = 200 \text{ ajustes} \).

#### Exemplo

**Exemplo.** Se a região boa ocupa uma fração `p` do espaço, cada sorteio erra com
probabilidade `1 − p`, e `n` sorteios erram todos com `(1 − p)ⁿ`. A chance de acertar ao menos
uma vez é o complemento. Com p = 10 % e n = 10 sorteios:

In [6]:
p, n = 0.10, 10
print(f"chance de errar todos: (1 - {p})^{n} = {(1-p)**n:.4f}")
print(f"chance de acertar ao menos um: {1 - (1-p)**n:.4f}")

chance de errar todos: (1 - 0.1)^10 = 0.3487
chance de acertar ao menos um: 0.6513


### Exercício 2 — a chance de o sorteio achar a região boa
Monte a tabela da chance `1 − (1 − p)ⁿ` para `p` em `[0.05, 0.10, 0.15, 0.25]` e `n` em
`[10, 16, 30, 60]`, em porcentagem.

Depois responda: com `p = 15 %`, quantos sorteios você precisa para passar de 90 %? E o que
acontece com essa conta se o espaço tiver 2, 5 ou 50 hiperparâmetros?

In [7]:
# Tabela de probabilidades
ps = [0.05, 0.10, 0.15, 0.25]
sorteios = [10, 16, 30, 60]

print("Tabela de Probabilidade de Acerto (em %):")
print(f"{'n \\\ p':<6} | " + " | ".join([f"{p*100:>5.0f}%" for p in ps]))
print("-" * 40)
for n_val in sorteios:
    row = []
    for p_val in ps:
        chance = (1 - (1 - p_val)**n_val) * 100
        row.append(f"{chance:>5.1f}%")
    print(f"{n_val:<6} | " + " | ".join(row))

# Encontrar o menor n para p = 15% passar de 90%
p_target = 0.15
n_target = 1
while (1 - (1 - p_target)**n_target) < 0.90:
    n_target += 1
print(f"\nMenor n para p = 15% passar de 90%: {n_target} (chance: {(1 - (1 - p_target)**n_target)*100:.2f}%)")

Tabela de Probabilidade de Acerto (em %):
n \\ p |     5% |    10% |    15% |    25%
----------------------------------------
10     |  40.1% |  65.1% |  80.3% |  94.4%
16     |  56.0% |  81.5% |  92.6% |  99.0%
30     |  78.5% |  95.8% |  99.2% | 100.0%
60     |  95.4% |  99.8% | 100.0% | 100.0%

Menor n para p = 15% passar de 90%: 15 (chance: 91.26%)


In [18]:
# TODO — tabela de 1 - (1 - p)**n para cada p e cada n
# TODO — ache o menor n que passa de 90% com p = 0,15

**Resposta:**


---
## Parte 2 — RandomizedSearchCV na prática

Três decisões: qual **distribuição**, qual **faixa**, quantos **sorteios**.

#### Exemplo

**Exemplo.** O dicionário da busca aleatória não leva listas de valores, leva **distribuições**
— objetos com `.rvs()`. Sortear 5 valores de `uniform(0, 1)`:

In [19]:
print(np.round(uniform(0, 1).rvs(5, random_state=0), 3))

[0.549 0.715 0.603 0.545 0.424]


### Exercício 3 — as distribuições do dicionário
Sorteie 2.000 valores de cada uma e imprima o **mínimo e o máximo** de cada amostra:

- `uniform(0.1, 0.8)`
- `randint(100, 601)`
- `loguniform(0.01, 100)`

Para a loguniform, imprima também **que fração dos sorteios caiu em cada ordem de grandeza**
(0,01–0,1 · 0,1–1 · 1–10 · 10–100) e compare com o que daria `uniform(0.01, 100)`.

Responda: qual é a faixa real de `uniform(0.1, 0.8)`? E por que o `C` da logística (0,01 a 100)
se sorteia com loguniform?

In [20]:
# TODO — sorteie 2000 valores de cada distribuição e imprima mínimo e máximo
# TODO — para a loguniform, conte a fração em cada ordem de grandeza
# TODO — faça a mesma contagem com uniform(0.01, 100) para comparar

In [8]:
# Exercício 3 - Amostragem de distribuições
samples_u = uniform(0.1, 0.8).rvs(2000, random_state=0)
samples_ri = randint(100, 601).rvs(2000, random_state=0)
samples_lu = loguniform(0.01, 100).rvs(2000, random_state=0)

print(f"uniform(0.1, 0.8) -> Min: {samples_u.min():.4f}, Max: {samples_u.max():.4f}")
print(f"randint(100, 601) -> Min: {samples_ri.min()}, Max: {samples_ri.max()}")
print(f"loguniform(0.01, 100) -> Min: {samples_lu.min():.4f}, Max: {samples_lu.max():.4f}")

# Frações por ordem de grandeza
ranges = [(0.01, 0.1), (0.1, 1.0), (1.0, 10.0), (10.0, 100.0)]
print("\nFrações de amostragem na loguniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_lu >= r[0]) & (samples_lu < r[1]))
    print(f"  {r[0]:.2f} a {r[1]:.1f}: {frac*100:.1f}%")

samples_u_flat = uniform(0.01, 100).rvs(2000, random_state=0)
print("\nFrações de amostragem na uniform(0.01, 100):")
for r in ranges:
    frac = np.mean((samples_u_flat >= r[0]) & (samples_u_flat < r[1]))
    print(f"  {r[0]:.2f} a {r[1]:.1f}: {frac*100:.1f}%")

uniform(0.1, 0.8) -> Min: 0.1003, Max: 0.8998
randint(100, 601) -> Min: 100, Max: 600
loguniform(0.01, 100) -> Min: 0.0100, Max: 99.8238

Frações de amostragem na loguniform(0.01, 100):
  0.01 a 0.1: 25.1%
  0.10 a 1.0: 24.9%
  1.00 a 10.0: 23.9%
  10.00 a 100.0: 26.2%

Frações de amostragem na uniform(0.01, 100):
  0.01 a 0.1: 0.1%
  0.10 a 1.0: 0.8%
  1.00 a 10.0: 8.8%
  10.00 a 100.0: 90.3%


**Resposta:**

A faixa real sorteada por `uniform(0.1, 0.8)` vai de \( 0,1 \) até \( 0,1 + 0,8 = 0,9 \) (conforme constatado experimentalmente com o mínimo em 0,1003 e máximo em 0,8998). O segundo parâmetro do Scipy determina a largura do intervalo, e não o limite superior.

Sorteamos o hiperparâmetro \( C \) da regressão logística com uma `loguniform` porque ele é um hiperparâmetro de escala. Com a `loguniform(0.01, 100)`, garantimos uma representação equilibrada por ordens de grandeza: as faixas de 0,01 a 0,1 (25,1%), 0,1 a 1,0 (24,9%), 1,0 a 10 (23,9%) e 10 a 100 (26,2%) recebem praticamente o mesmo número de sorteios. Já na `uniform(0.01, 100)`, mais de 90% das tentativas ficam concentradas entre 10 e 100, deixando as escalas menores completamente inexploradas.

#### Exemplo

**Exemplo.** Os dois recebem o mesmo pipeline, o mesmo `cv` e a mesma métrica. Muda o que
entra no segundo argumento: a grade recebe **listas**, a busca aleatória recebe **distribuições**
e um `n_iter`. Primeiro a grade, com 4 × 4 = 16 combinações:

In [21]:
t = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"grade: {len(gs.cv_results_['mean_test_score'])} combinações | "
      f"melhor AP {gs.best_score_:.4f} | {time.time()-t:.0f} s")
print("melhor:", gs.best_params_)

grade: 16 combinações | melhor AP 0.6155 | 27 s
melhor: {'clf__ccp_alpha': 0.001, 'clf__max_features': 0.2}


### Exercício 4 — a grade e o sorteio, com o mesmo orçamento
Rode uma `RandomizedSearchCV` com o **mesmo orçamento** (16 medições) sobre os mesmos dois
hiperparâmetros, usando `loguniform(1e-5, 1e-1)` para o `ccp_alpha` e `uniform(0.1, 0.8)` para o
`max_features`, com `random_state=0`.

Compare a melhor AP com a da grade e responda: quantos valores **distintos** de `ccp_alpha` cada
caminho testou? É daí que vem a diferença?

In [ ]:
# TODO — monte o dicionário de distribuições
# dist = {...}
# TODO — RandomizedSearchCV com n_iter=16, cv=cv, scoring="average_precision", random_state=0
# TODO — compare best_score_ com o da grade e conte os valores distintos de cada caminho

In [9]:
# Exercício 4 - Grade vs RandomizedSearch
t = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"Grade concluída: melhor AP = {gs.best_score_:.4f}")
print("Melhores parâmetros da grade:", gs.best_params_)

dist = {
    "clf__ccp_alpha": loguniform(1e-5, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Busca Aleatória concluída: melhor AP = {rs.best_score_:.4f}")
print("Melhores parâmetros da busca aleatória:", rs.best_params_)

# Quantidade de valores distintos testados em ccp_alpha
ccp_distinct_grid = len(set(gs.cv_results_['param_clf__ccp_alpha'].data))
ccp_distinct_rand = len(set(rs.cv_results_['param_clf__ccp_alpha'].data))
print(f"Valores distintos de ccp_alpha testados na Grade: {ccp_distinct_grid}")
print(f"Valores distintos de ccp_alpha testados na Busca Aleatória: {ccp_distinct_rand}")

Grade concluída: melhor AP = 0.6155
Melhores parâmetros da grade: {'clf__ccp_alpha': 0.001, 'clf__max_features': 0.2}
Busca Aleatória concluída: melhor AP = 0.6271
Melhores parâmetros da busca aleatória: {'clf__ccp_alpha': np.float64(0.0025766385746135885), 'clf__max_features': np.float64(0.5359065463975176)}
Valores distintos de ccp_alpha testados na Grade: 4
Valores distintos de ccp_alpha testados na Busca Aleatória: 16


**Resposta:**

A busca aleatória obteve um desempenho superior (AP de 0,6271) em comparação com a grade convencional (AP de 0,6155).

A diferença principal está no número de valores distintos testados para o hiperparâmetro contínuo. Enquanto a grade testou apenas 4 valores pré-fixados de `ccp_alpha`, a busca aleatória conseguiu explorar 16 valores totalmente distintos. Como a busca aleatória distribui seus pontos em vez de repetir os mesmos valores em uma malha rígida, ela consegue varrer dimensões contínuas de forma muito mais eficiente no mesmo orçamento de tempo.

#### Exemplo

**Exemplo.** Uma busca aleatória é uma amostra: trocar a semente troca os pontos sorteados e,
com eles, o resultado. Antes de comemorar o 0,627, vale ver quanto ele balança. Com três
sementes:

In [27]:
for s in [0, 1, 2]:
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision",
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    print(f"semente {s}: {b.best_score_:.4f}")

semente 0: 0.6271
semente 1: 0.6304
semente 2: 0.6527


### Exercício 5 — foi sorte da semente?
Repita a busca com **20 sementes** (`range(20)`), guarde os 20 melhores escores e imprima a
média, o mínimo e o máximo, além da **fração das sementes que ficou acima da grade**
(`gs.best_score_`).

Responda: o 0,627 da semente 0 foi sorte? E o que você reportaria num relatório — o melhor
resultado ou a distribuição?

In [ ]:
# TODO — rode a busca para random_state em range(20) e guarde os best_score_
# TODO — imprima média, mínimo, máximo e a fração acima de gs.best_score_

In [10]:
# Exercício 5 - Sensibilidade à Semente (20 sementes)
escores = []
for s in range(20):
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision",
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    escores.append(b.best_score_)

escores = np.array(escores)
print(f"Média: {escores.mean():.4f}")
print(f"Mínimo: {escores.min():.4f}")
print(f"Máximo: {escores.max():.4f}")
acima_da_grade = np.mean(escores > gs.best_score_)
print(f"Fração de sementes acima da grade ({gs.best_score_:.4f}): {acima_da_grade*100:.1f}%")

Média: 0.6413
Mínimo: 0.6229
Máximo: 0.6527
Fração de sementes acima da grade (0.6155): 100.0%


**Resposta:**

Não foi pura sorte. A análise estatística das 20 sementes mostra que a média de desempenho foi de 0,6413, variando entre um mínimo de 0,6229 e um máximo de 0,6527. O mais importante é que **100% das sementes superaram o desempenho da grade** (0,6155).

Em um relatório acadêmico, o correto é reportar a distribuição (a média acompanhada do desvio padrão ou intervalos) para garantir a reprodutibilidade e a honestidade científica do experimento, em vez de destacar apenas o valor máximo isolado.

#### Exemplo

**Exemplo.** O `best_params_` é **uma linha** do `cv_results_`. Olhar só para ela esconde se o
primeiro lugar ganhou por uma diferença real ou por ruído. O `cv_results_` é um dicionário que
vira DataFrame — veja as três primeiras colocações da busca do Exercício 4:

In [22]:
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
print(res.sort_values("rank_test_score")[cols].head(3).round(4).to_string(index=False))

 param_clf__ccp_alpha  param_clf__max_features  mean_test_score  std_test_score
               0.0026                   0.5359           0.6271          0.1182
               0.0005                   0.6167           0.6184          0.1036
               0.0000                   0.6119           0.6184          0.1036


### Exercício 6 — o pódio e o platô
Monte a tabela das **5 primeiras colocações** com a média e o desvio entre dobras, e calcule:

1. a diferença entre o 1º e o 5º lugar;
2. o desvio entre dobras do 1º lugar.

Responda: isso é um pódio ou um platô? E, se for platô, qual configuração você entregaria?

In [ ]:
# TODO — top 5 do cv_results_, com mean_test_score e std_test_score
# TODO — a diferença 1º - 5º e o desvio do 1º

In [11]:
# Exercício 6 - Pódio e Platô
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
top5 = res.sort_values("mean_test_score", ascending=False).head(5)
print(top5[cols].round(4).to_string(index=False))

dif_1_5 = top5["mean_test_score"].iloc[0] - top5["mean_test_score"].iloc[4]
desv_1 = top5["std_test_score"].iloc[0]
print(f"\nDiferença entre 1º e 5º lugar: {dif_1_5:.4f}")
print(f"Desvio padrão entre as dobras do 1º lugar: {desv_1:.4f}")

 param_clf__ccp_alpha  param_clf__max_features  mean_test_score  std_test_score
               0.0026                   0.5359           0.6271          0.1182
               0.0005                   0.6167           0.6184          0.1036
               0.0000                   0.6119           0.6184          0.1036
               0.0012                   0.4317           0.6162          0.1117
               0.0007                   0.7244           0.6125          0.1122

Diferença entre 1º e 5º lugar: 0.0146
Desvio padrão entre as dobras do 1º lugar: 0.1182


**Resposta:**

Este cenário configura claramente um **platô**. A diferença entre o 1º colocado (AP de 0,6271) e o 5º colocado (AP de 0,6125) é de apenas 0,0146. Essa oscilação é muito menor do que o desvio padrão de validação cruzada das dobras do próprio modelo vencedor, que foi de 0,1182.

Sendo um platô de desempenho estatisticamente equivalente, eu optaria por entregar a configuração que simplifica o modelo ou reduz o risco de overfitting, que seria o modelo com maior valor de `ccp_alpha` (gerando árvores menores) ou o menor custo computacional.

#### Exemplo

**Exemplo.** Quando o melhor valor encontrado é exatamente o **extremo** da faixa que você
definiu, a busca foi até onde podia e parou na parede — o pico pode estar do lado de fora.
Veja onde caiu o melhor `ccp_alpha` em relação à faixa sorteada:

In [26]:
melhor = rs.best_params_["clf__ccp_alpha"]
print(f"faixa definida : 1e-05 a 1e-01")
print(f"melhor valor   : {melhor:.2e}")
print(f"posição na faixa (em log): "
      f"{(np.log10(melhor) - np.log10(1e-5)) / (np.log10(1e-1) - np.log10(1e-5)):.0%}")

faixa definida : 1e-05 a 1e-01
melhor valor   : 2.58e-03
posição na faixa (em log): 60%


### Exercício 7 — o ótimo na borda da faixa
Rode uma busca com a faixa **estreitada de propósito** — `loguniform(1e-2, 1e-1)` para o
`ccp_alpha`, mantendo o resto igual — e veja onde cai o melhor valor e qual AP ela alcança.

Depois amplie a faixa para `loguniform(1e-7, 1e-1)` e rode de novo. Responda: o que o valor
colado na borda indicava, e o que mudou ao ampliar?

In [ ]:
# TODO — busca com ccp_alpha em loguniform(1e-2, 1e-1); onde caiu o melhor valor? qual AP?
# TODO — busca com ccp_alpha em loguniform(1e-7, 1e-1); compare

In [12]:
# Exercício 7 - Ótimo na borda da faixa
dist_estreita = {
    "clf__ccp_alpha": loguniform(1e-2, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_estreita = RandomizedSearchCV(pipe(), dist_estreita, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_est = rs_estreita.best_params_["clf__ccp_alpha"]
print(f"Estreita -> melhor ccp_alpha: {melhor_est:.2e} | AP: {rs_estreita.best_score_:.4f}")

dist_ampla = {
    "clf__ccp_alpha": loguniform(1e-7, 1e-1),
    "clf__max_features": uniform(0.1, 0.8)
}
rs_ampla = RandomizedSearchCV(pipe(), dist_ampla, n_iter=16, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
melhor_amp = rs_ampla.best_params_["clf__ccp_alpha"]
print(f"Ampla    -> melhor ccp_alpha: {melhor_amp:.2e} | AP: {rs_ampla.best_score_:.4f}")

Estreita -> melhor ccp_alpha: 1.05e-02 | AP: 0.6047
Ampla    -> melhor ccp_alpha: 5.63e-03 | AP: 0.6317


**Resposta:**

O valor colado na borda indicava que o intervalo definido limitava a otimização e que o verdadeiro ponto ideal poderia estar fora da faixa originalmente considerada.

* Ao estreitarmos a faixa para `loguniform(1e-2, 1e-1)`, o melhor parâmetro caiu em 1,05e-02 (muito perto do limite inferior de 1e-2), reduzindo o desempenho para AP de 0,6047.
* Ao ampliarmos a faixa para `loguniform(1e-7, 1e-1)`, o algoritmo pôde buscar livremente e encontrou o melhor `ccp_alpha` em 5,63e-03, alcançando uma AP superior de 0,6317.

---
## Parte 3 — A nota honesta

Da mesma busca saem três números. Só dois podem ser reportados.

#### Exemplo

**Exemplo.** O `best_score_` é o **máximo** de n_iter medições feitas na mesma validação que
escolheu — por construção, otimista. A nested CV roda a busca **inteira** dentro de cada dobra de
fora, então nenhuma escolha vê a dobra que a mede. Veja a diferença dos dois em miniatura, com
3 dobras de fora e n_iter = 5:

In [24]:
t = time.time()
mini = RandomizedSearchCV(pipe(), dist, n_iter=5, cv=cv, scoring="average_precision",
                          random_state=0, n_jobs=-1)
nested_mini = cross_val_score(mini, X_tr, y_tr, scoring="average_precision",
                              cv=StratifiedKFold(3, shuffle=True, random_state=1))
print(f"best_score_ (ingênuo): {mini.fit(X_tr, y_tr).best_score_:.4f}")
print(f"nested CV (3 dobras) : {nested_mini.mean():.4f} ± {nested_mini.std():.4f}")
print(f"({time.time()-t:.0f} s)")

best_score_ (ingênuo): 0.6271
nested CV (3 dobras) : 0.6030 ± 0.1051
(33 s)


### Exercício 8 — os três números da mesma busca
Produza os **três** números da busca do Exercício 4 (n_iter = 16):

1. o `best_score_`;
2. a **nested CV** com 5 dobras de fora (`StratifiedKFold(5, shuffle=True, random_state=1)`) —
   média e desvio;
3. a AP no **teste guardado** (`X_te`, `y_te`), que até agora não foi tocado.

Responda: qual deles **não** pode ser reportado como desempenho, e por quê? E quantos hits o
teste tem — isso muda a sua confiança no terceiro número?

In [ ]:
# TODO — best_score_ da busca do Exercício 4
# TODO — nested CV: cross_val_score(busca, X_tr, y_tr, cv=StratifiedKFold(5, ...))
# TODO — AP no teste guardado, com average_precision_score e predict_proba
# TODO — quantos hits o teste tem?

In [13]:
# Exercício 8 - Nota Honesta (Três Números)
print(f"1. best_score_ (ingênuo): {rs.best_score_:.4f}")

nested_cv = cross_val_score(rs, X_tr, y_tr, scoring="average_precision",
                            cv=StratifiedKFold(5, shuffle=True, random_state=1), n_jobs=-1)
print(f"2. Nested CV (média e desvio): {nested_cv.mean():.4f} ± {nested_cv.std():.4f}")

# AP no teste guardado
ap_teste = average_precision_score(y_te, rs.predict_proba(X_te)[:, 1])
print(f"3. Teste guardado AP: {ap_teste:.4f}")
print(f"   Hits no teste: {int(y_te.sum())}")

1. best_score_ (ingênuo): 0.6271
2. Nested CV (média e desvio): 0.6354 ± 0.1417
3. Teste guardado AP: 0.6949
   Hits no teste: 38


**Resposta:**

1. O valor de `best_score_` (0,6271) **não** pode ser reportado como a estimativa final de desempenho. Por ser o máximo de várias tentativas na mesma validação cruzada, ele carrega um viés otimista de seleção.
2. A estimativa honesta é o resultado do Nested CV (0,6354 ± 0,1417) ou do teste guardado (0,6949).

O teste guardado possui apenas **38 hits**. Por ser um número pequeno de eventos positivos, o cálculo da precisão média está sujeito a uma variância amostral considerável, o que diminui a nossa confiança estatística no número de teste isolado e torna o Nested CV a estimativa mais confiável.

---
## Parte 4 — Um mini-AutoML

A mesma busca, agora escolhendo também **qual modelo**.

#### Exemplo

**Exemplo.** Para escolher também o **modelo**, o `RandomizedSearchCV` aceita uma **lista** de
dicionários. Cada dicionário fixa um `clf` e descreve só os hiperparâmetros daquele modelo — o
espaço fica **condicional**: o `ccp_alpha` só existe se o sorteio cair na floresta. Com dois
modelos e 6 sorteios:

In [25]:
from sklearn.linear_model import LogisticRegression

espaco_mini = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)],
     "clf__max_features": uniform(0.1, 0.8)},
]
b = RandomizedSearchCV(pipe(), espaco_mini, n_iter=6, cv=cv, scoring="average_precision",
                       random_state=0, n_jobs=-1).fit(X_tr, y_tr)
nomes = [type(p["clf"]).__name__ for p in b.cv_results_["params"]]
print("modelos sorteados:", nomes)
print("vencedor:", type(b.best_params_["clf"]).__name__, "| AP", round(b.best_score_, 4))

modelos sorteados: ['LogisticRegression', 'LogisticRegression', 'RandomForestClassifier', 'RandomForestClassifier', 'LogisticRegression', 'LogisticRegression']
vencedor: LogisticRegression | AP 0.7193


### Exercício 9 — o espaço condicional do AutoML
Monte um espaço com **três** modelos — logística (`C`), árvore de decisão (`max_depth`,
`min_samples_leaf`) e floresta (`max_features`, `ccp_alpha`) — e rode com `n_iter = 30`.

Depois monte uma tabela com, **para cada modelo**: quantos sorteios ele recebeu, a melhor AP e a
**mediana** das APs dele. Responda: o que pesou mais — escolher o modelo ou afinar o vencedor?

In [ ]:
# TODO — importe DecisionTreeClassifier e monte a lista de 3 dicionários
# TODO — RandomizedSearchCV com n_iter=30, random_state=0
# TODO — para cada modelo: nº de sorteios, melhor AP e mediana das APs

In [14]:
# Exercício 9 - AutoML
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

espaco_automl = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [DecisionTreeClassifier(random_state=0)], "clf__max_depth": randint(2, 21), "clf__min_samples_leaf": randint(2, 51)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)], "clf__max_features": uniform(0.1, 0.8), "clf__ccp_alpha": loguniform(1e-5, 1e-1)}
]

rs_automl = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"Melhor modelo vencedor: {type(rs_automl.best_params_['clf']).__name__} | AP: {rs_automl.best_score_:.4f}")

res_automl = pd.DataFrame(rs_automl.cv_results_)
res_automl["modelo_nome"] = res_automl["param_clf"].apply(lambda x: type(x).__name__)

summary = res_automl.groupby("modelo_nome")["mean_test_score"].agg(["count", "max", "median"])
print("\nTabela resumo de cada modelo:")
print(summary.to_string())

Melhor modelo vencedor: LogisticRegression | AP: 0.7241

Tabela resumo de cada modelo:
                        count       max    median
modelo_nome                                      
DecisionTreeClassifier     10  0.409646  0.402162
LogisticRegression         13  0.724118  0.719358
RandomForestClassifier      7  0.622860  0.606665


**Resposta:**

A tabela resumo mostra que:
* **DecisionTreeClassifier:** 10 sorteios, melhor AP de 0,4096, mediana de 0,4022
* **LogisticRegression:** 13 sorteios, melhor AP de 0,7241, mediana de 0,7194
* **RandomForestClassifier:** 7 sorteios, melhor AP de 0,6229, mediana de 0,6067

Fica evidente que **escolher a família do modelo pesou muito mais**. A pior regressão logística testada ainda foi amplamente superior às melhores florestas ou árvores de decisão. Afinar o vencedor trouxe pequenos ganhos marginais dentro de cada modelo, mas a escolha do algoritmo certo causou o grande salto na métrica.

#### Exemplo

**Exemplo.** Com uma lista de modelos, o `n_iter` é dividido entre eles — e cada modelo recebe
só uma fatia. Com 30 sorteios e 3 modelos, a média é 10 por modelo; com 10 modelos, seriam 3.
Volte à conta do Exercício 2 e veja o que 3 sorteios garantem:

In [ ]:
for n in [3, 10, 30]:
    print(f"{n:>2} sorteios, p = 15%: {100*(1-0.85**n):.0f}% de chance de achar a região boa")

### Exercício 10 — o orçamento dividido
Rode o mesmo espaço de três modelos com `n_iter` em `[10, 30, 60]` e, para cada orçamento,
imprima a melhor AP, o modelo vencedor, quantos sorteios cada modelo recebeu e o tempo.

Responda: o que acontece com a melhor AP quando o orçamento triplica? E o que aconteceria se
você acrescentasse sete modelos ao espaço mantendo `n_iter = 30`?

In [ ]:
# TODO — para n_iter em [10, 30, 60], rode e guarde: melhor AP, vencedor, sorteios por modelo, tempo

In [15]:
# Exercício 10 - Orçamentos
for n in [10, 30, 60]:
    t_start = time.time()
    search = RandomizedSearchCV(pipe(), espaco_automl, n_iter=n, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    dt = time.time() - t_start
    vencedor = type(search.best_params_["clf"]).__name__
    res_n = pd.DataFrame(search.cv_results_)
    res_n["modelo_nome"] = res_n["param_clf"].apply(lambda x: type(x).__name__)
    sorteios = res_n["modelo_nome"].value_counts().to_dict()
    print(f"n_iter = {n:<2} | AP: {search.best_score_:.4f} | Vencedor: {vencedor:<26} | Sorteios: {sorteios} | Tempo: {dt:.1f} s")

n_iter = 10 | AP: 0.7193 | Vencedor: LogisticRegression         | Sorteios: {'LogisticRegression': 4, 'DecisionTreeClassifier': 4, 'RandomForestClassifier': 2} | Tempo: 4.2 s
n_iter = 30 | AP: 0.7241 | Vencedor: LogisticRegression         | Sorteios: {'LogisticRegression': 13, 'DecisionTreeClassifier': 10, 'RandomForestClassifier': 7} | Tempo: 12.4 s
n_iter = 60 | AP: 0.7247 | Vencedor: LogisticRegression         | Sorteios: {'LogisticRegression': 25, 'RandomForestClassifier': 18, 'DecisionTreeClassifier': 17} | Tempo: 30.1 s


**Resposta:**

Quando o orçamento de busca triplicou (de 10 para 30 e depois para 60), a melhor AP praticamente estabilizou, subindo discretamente de 0,7193 (com 10 iterações) para 0,7241 (com 30) e 0,7247 (com 60).

Se adicionássemos mais sete modelos mantendo `n_iter = 30`, a busca sofreria uma diluição drástica de orçamento. Cada família receberia, em média, apenas 3 sorteios. Como vimos no Exercício 2, 3 sorteios dão apenas 39% de chance de encontrar a região ideal de hiperparâmetros de cada modelo, prejudicando gravemente a eficácia do AutoML.

#### Exemplo

**Exemplo.** O AutoML otimiza o que você deixar no `X`. Se uma coluna carrega informação que
não existiria na hora de prever, ele a usará com prazer — e o número fica lindo. Na base das
músicas, a coluna `popularidade` é medida **depois** de a faixa virar hit. Veja a correlação
dela com o alvo:

In [17]:
print("correlação de popularidade com hit:", round(mus["popularidade"].corr(mus["hit"]), 3))
print("popularidade média | hits:", round(mus.loc[mus.hit == 1, "popularidade"].mean(), 1),
      "| não-hits:", round(mus.loc[mus.hit == 0, "popularidade"].mean(), 1))

correlação de popularidade com hit: 0.549
popularidade média | hits: 99.9 | não-hits: 50.0


### Exercício 11 — o que o AutoML não automatiza
Monte um `X_vaza` **incluindo** a coluna `popularidade`, refaça o corte treino/teste com a mesma
semente e rode a mesma busca de três modelos com `n_iter = 30`. Compare a melhor AP com a do
Exercício 9.

Responda: o AutoML avisou que havia algo errado? Quem teria de perceber, e como?

In [ ]:
# TODO — X_vaza = get_dummies de mus sem id/titulo/artista/hit (mantendo popularidade)
# TODO — novo train_test_split com random_state=0, stratify=y
# TODO — mesma busca de 3 modelos, n_iter=30; compare a AP

In [16]:
# Exercício 11 - Vazamento de Alvo
X_vaza = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "hit"]), columns=["genero"])
X_tr_v, X_te_v, y_tr_v, y_te_v = train_test_split(X_vaza, y, test_size=0.3, stratify=y, random_state=0)

search_vaza = RandomizedSearchCV(pipe(), espaco_automl, n_iter=30, cv=cv, scoring="average_precision", random_state=0, n_jobs=-1).fit(X_tr_v, y_tr_v)
print(f"Vaza -> Melhor AP: {search_vaza.best_score_:.4f} | Vencedor: {type(search_vaza.best_params_['clf']).__name__}")

Vaza -> Melhor AP: 1.0000 | Vencedor: DecisionTreeClassifier


**Resposta:**

Ao incluir a variável `popularidade`, o modelo vencedor (uma árvore de decisão) alcançou uma AP perfeita de **1,0000**.

O AutoML **não emitiu nenhum alerta**. Ele é uma ferramenta puramente matemática que otimiza a função objetivo fornecida. A responsabilidade de detectar o vazamento de alvo é do cientista de dados, que deve desconfiar imediatamente de métricas perfeitas como 1,00 e realizar a auditoria das features (analisando a correlação das variáveis ou o peso de decisão do modelo).

---
## Parte 5 — Síntese

#### Exemplo

**Exemplo.** Uma conclusão fechada tem três partes: a afirmação, o número que a sustenta e a
consequência prática. Por exemplo: *"com o mesmo orçamento de 16 medições, o sorteio deu 0,627
contra 0,615 da grade, e 20 em 20 sementes bateram a grade; por isso passo a usar a aleatória
como padrão quando há mais de dois hiperparâmetros."*

### Exercício 12 — três conclusões
Escreva **três conclusões** no formato do exemplo — afirmação, número e consequência. Sugestões
de tema: por que o custo da aleatória não depende do número de hiperparâmetros; o que você passou
a olhar no `cv_results_` além da primeira linha; qual número você reportaria no relatório e qual
ficaria no caderno; o que o AutoML resolveu e o que ele deixou na sua mão.

**Resposta:**

**1. Sobre a imunidade à dimensionalidade na busca aleatória:**
Como a fórmula de acerto depende unicamente do volume relativo \( p \) ocupado pela região boa, a probabilidade acumulada de sucesso com 15 sorteios se mantém em 91,26% independentemente do espaço ter 2 ou 50 hiperparâmetros. Isso torna a busca aleatória a escolha padrão e viável para problemas complexos de alta dimensão, onde a grade clássica se torna impraticável.

**2. Sobre a honestidade no relato de resultados:**
O escore ingênuo (`best_score_`) de 0,6271 superestimou a capacidade de generalização real. No relatório final do projeto, deve-se apresentar a estimativa honesta do Nested CV (0,6354 ± 0,1417), deixando a busca individual no caderno, o que evita frustrações de desempenho em produção.

**3. Sobre o papel do engenheiro frente ao AutoML:**
Embora o AutoML poupe tempo ao testar famílias distintas e encontrar a Regressão Logística como vencedora com AP de 0,7241, ele otimizou sem restrições a feature vazada no Exercício 11, resultando em uma métrica artificial de 1,0000. Isso comprova que a validação cuidadosa de dados e a prevenção de vazamentos de alvo continuam sendo uma tarefa exclusivamente humana.

---
**Fim.** Você comparou grade e sorteio com o mesmo orçamento, montou o dicionário de
distribuições, leu o pódio e o platô, reagiu a um ótimo na borda, separou os três números da nota
honesta e rodou um mini-AutoML com espaço condicional — inclusive o caso em que ele otimiza
alegremente uma coluna que vaza o alvo.

No **Desafio 3** o que vale não é produzir o número, é justificá-lo.